In [1]:
import plotly.express as px
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np

In [2]:
style = "2"
results_path: str = f'../src/optimal_explorer/strategies/combination_lock/logs/game_results/style{style}.jsonl'
llm_call_path: str = f'../src/optimal_explorer/strategies/combination_lock/logs/llm_calls/style{style}_'
bayes_optimal_path: str = f'../src/optimal_explorer/strategies/combination_lock/logs/bayes_optimal.jsonl'

models = [
    "Gemini Pro 2.5",
    "DeepSeek R1",
    "Claude Opus 4",
    "Claude 3.5 Sonnet",
    "OpenAI o3",
]

model_ids = [
    "google/gemini-2.5-pro-preview",
    "deepseek/deepseek-r1-0528",
    "anthropic/claude-opus-4",
    "anthropic/claude-3.5-sonnet",
    "openai/o3",
]

llm_call_file_ids = [
    "gemini-2.5-pro-preview",
    "deepseek-r1-0528",
    "claude-opus-4",
    "claude-3.5-sonnet",
    "o3",
]

In [3]:
tokens_used = []

for i, model in enumerate(models):
    print(f"Processing {model}...")
    with open(f'{llm_call_path}{llm_call_file_ids[i]}.jsonl', 'r') as f:
        for line in f:
            data = json.loads(line)
            model_name = data['model']
            try:
                tokens = data['usage']['total_tokens']
            except:
                tokens = 0
            game_id = data['game_id']
            style_val = data['prompt_style']
            user_prompt = data['user_prompt']
            # number of times the word "Attempt" appears in the user prompt
            attempt = user_prompt.count("Attempt")
            tokens_used.append({
                'model': model_name,
                'tokens': tokens,
                'game_id': game_id,
                'style': style_val,
                'attempt': attempt
            })

tokens_used_df = pd.DataFrame(tokens_used)

Processing Gemini Pro 2.5...
Processing DeepSeek R1...
Processing Claude Opus 4...
Processing Claude 3.5 Sonnet...
Processing OpenAI o3...


In [11]:
tokens_used_df.head(2)

,model,tokens,game_id,style,attempt
0,google/gemini-2.5-pro-preview,519,65,2,0
1,google/gemini-2.5-pro-preview,549,85,2,0


In [39]:
tokens_model_attempt = []
tokens_model_attempt_var = []

for i, model in enumerate(models):
    print(f"Processing {model}...")
    model_id = model_ids[i]
    games_df = tokens_used_df[ 
        (tokens_used_df['model'] == model_id) &
        (tokens_used_df['style'].astype(str) == style)
    ]
    tokens_stats_by_attempt = games_df[games_df['tokens'] != 0]\
        .groupby('attempt')['tokens'].agg(['mean', 'var']).reset_index()
    tokens_model_attempt.append(tokens_stats_by_attempt['mean'].tolist())
    tokens_model_attempt_var.append(tokens_stats_by_attempt['var'].tolist())

Processing Gemini Pro 2.5...
Processing DeepSeek R1...
Processing Claude Opus 4...
Processing Claude 3.5 Sonnet...
Processing OpenAI o3...


In [43]:
fig = go.Figure()
colors = [px.colors.qualitative.Dark24[i] for i in [1, 10, 6, 15, 19]]
# Helper to convert hex color to rgba with alpha
def hex_to_rgba(hex_color, alpha=0.2):
    hex_color = hex_color.lstrip('#')
    if len(hex_color) == 6:
        r, g, b = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    else:
        # fallback to black if color is not valid
        r, g, b = 0, 0, 0
    return f'rgba({r},{g},{b},{alpha})'

for i, model in enumerate(models):
    x = list(range(len(tokens_model_attempt[i])))
    y = tokens_model_attempt[i]
    # Get standard deviation from variance, handle possible NaN
    var = tokens_model_attempt_var[i]
    std = [v**0.5 if v is not None and not pd.isna(v) else 0 for v in var]
    y_upper = [a + b for a, b in zip(y, std)]
    y_lower = [a - b for a, b in zip(y, std)]

    # Add shaded error band
    fig.add_trace(go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        fillcolor=hex_to_rgba(colors[i], 0.2),  # use rgba for transparency
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False,
        name=f"{model} error",
    ))

    # Add mean line
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        name=model,
        line=dict(color=colors[i], width=4),
        marker=dict(size=10),
    ))
fig.update_layout(
    title='',
    xaxis_title='Episode',
    yaxis_title='Tokens Used (Total)',
    legend_title='Model',
    template='plotly_white',
    width=650,
    height=450,
)